# Spark Session Initialization

Initialize the Spark Session used for all DataFrame operations in this notebook.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# os.environ["PYSPARK_PYTHON"] = sys.executable
# os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

builder = ( SparkSession.builder \
    .appName("BGG Data Validation") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.LocalLogStore")
            
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# Paths Configuration

Define all input and output paths used in this notebook.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]
DATA_PATH = PROJECT_ROOT / "data"

# BRONZE_PATH = DATA_PATH / "bronze"
SOURCE_PATH = DATA_PATH / "source"
GAMES_BGG_CSV = SOURCE_PATH / "bgg_db_4999_shortened.csv"

# DataFrame Schemas Definition

Define structured schemas for:
- bgg_games
- customers
- employees
- sales

In [ ]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, LongType, StringType, DoubleType, TimestampType, DateType
)


games_bgg_schema = StructType([
    StructField("rank", IntegerType(), True),
    StructField("bgg_url", StringType(), False),
    StructField("game_id", IntegerType(), False),
    StructField("names", StringType(), False),
    StructField("min_players", IntegerType(), True),
    StructField("max_players", IntegerType(), True),
    StructField("avg_time", IntegerType(), True),
    StructField("min_time", IntegerType(), True),
    StructField("max_time", IntegerType(), True),
    StructField("year", IntegerType(), True),
    StructField("avg_rating", DoubleType(), True),
    StructField("geek_rating", DoubleType(), True),
    StructField("num_votes", IntegerType(), True),
    StructField("image_url", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("mechanic", StringType(), True),
    StructField("owned", IntegerType(), True),
    StructField("category", StringType(), True),
    StructField("designer", StringType(), True),
    StructField("weight", DoubleType(), True),
])


customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("first_name", StringType(), False),
    StructField("last_name", StringType(), False),
    StructField("email", StringType(), False),
    StructField("country_id", IntegerType(), False),
    StructField("registration_date", DateType(), False),
    StructField("ingestion_timestamp", TimestampType(), False),
    StructField("source_system", StringType(), False),
])


employees_schema = StructType([
    StructField("employee_id", StringType(), False),
    StructField("first_name", StringType(), False),
    StructField("last_name", StringType(), False),
    StructField("email", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("hire_date", DateType(), True),
    StructField("birth_date", DateType(), True),
    StructField("country_id", IntegerType(), False),
    StructField("ingestion_timestamp", TimestampType(), False),
    StructField("source_system", StringType(), False),
])


sales_schema = StructType([
    StructField("sale_id", LongType(), False),
    StructField("sale_timestamp", TimestampType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("game_id", IntegerType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_cost", DoubleType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("currency_code", StringType(), False),
    StructField("payment_method_code", StringType(), False),
    StructField("delivery_id", StringType(), False),
    StructField("employee_id", StringType(), False),
    StructField("vendor_id", StringType(), False),
    StructField("ga_id", StringType(), False),
    StructField("ingestion_timestamp", TimestampType(), False),
    StructField("source_system", StringType(), False),
])

# Load BoardGameGeek Games CSV

Read the shortened BGG games [dataset](https://www.kaggle.com/datasets/threnjen/board-games-database-from-boardgamegeek?select=games.csv) CSV from Kaggle into a Spark DataFrame.

In [ ]:
from pyspark.sql.functions import current_timestamp, lit

games_bgg_df = (
    spark.read
    .option("header", True)
    .schema(games_bgg_schema)
    .csv(str(GAMES_BGG_CSV))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("bgg.com"))
)

# Generate Random Customers and Employees 

Create synthetic customer and employee records for bronze Delta tables using Faker python library.

## Customers

In [ ]:
from utils.data_io import generate_customers

customer_rows = generate_customers(num_customers=2000, spark_connector=spark)
customers_df = spark.createDataFrame(customer_rows, schema=customers_schema)

In [ ]:
customers_df.show(5)

## Employees

In [ ]:
from utils.data_io import generate_employees

employees_rows = generate_employees(num_employees=12, spark_connector=spark)
employees_df = spark.createDataFrame(employees_rows,schema=employees_schema)

In [ ]:
employees_df.show(5)

# Save Bronze Tables

Writing to `delta` bronze-level tables:
- bgg_games
- customers
- employees

In [ ]:
from utils.data_io import save_to_bronze

save_to_bronze(games_bgg_df, "bgg_games")
save_to_bronze(customers_df, "customers")
save_to_bronze(employees_df, "employees")

# Generate and Save Random Sales

In [ ]:
from utils.data_io import generate_sales

sales_rows = generate_sales(10000, spark)

sales_df = spark.createDataFrame(
    sales_rows,
    schema=sales_schema
)

In [ ]:
save_to_bronze(sales_df, "sales")